# **Using llm-judge**

In [1]:
from llm_judge import Judge, JudgeConfig, annotate_csv

import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler("log.txt", mode="a", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)

## **Example Data**

Consider the following example data:
- `src` is our source text,
- `mt` is our machine translation that we want to rate in terms of quality,
- `src_lang` is the language of the source text,
- and `mt_lang` is the target language of the machine translation.

In [2]:
src = (
    "The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. "
    "I can see this feature as useful in an office setting with houseplants or if on vacation."
)
mt = (
    "Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. "
    "Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub."
)
src_lang = "English"
mt_lang = "German"

## **Setting up OpenAI, Anthropic, Google and Ollama Models**

The `Judge` constructor accepts five possible arguments:
- ```python
  openai: str | bool | None = None
  ```
- ```python
  anthropic: str | bool | None = None
  ```
- ```python
  google: str | bool | None = None
  ```
- ```python
  ollama: str | bool | None = None
  ```
- ...

All of these arguments can either be `str`, `bool`, or `None` (default):
- If `None` or `True` the constructor will try to read the corresponding API keys from your environment and throw an error if it can't find it in the environment.
- If `str` the constructor expects a valid API key in string format (with the exception of `ollama` which expects a valid model identifier).
- If `False` the specific model will be disabled in the judge.

The fifth possible argument is:
- ```python
  config: JudgeConfig | str | None = None
  ```

This can either be a `JudgeConfig` (see documentation), a `str` which is expected to be the path to a valid configuration file in [TOML](https://toml.io/) format, or `None` (default) in which case the default configuration is loaded.

**If `config` is given (not `None`) then `ollama` must not be a string!**

### **Example: Reading the API key for OpenAI model access from the environment**

In [3]:
judge_no_env = Judge(openai=True, anthropic=False, google=False, ollama=False)

ERROR:llm_judge._judge:Could not get an API key for OpenAI! Searched for 'OPENAI_API_KEY'.


RuntimeError: Could not get an API key for OpenAI! Searched for 'OPENAI_API_KEY'.

An error is thrown because the environment does not contain an API key for OpenAI models.

**Please never commit your environments or API keys!**

### **Example: Manually specifying the API key for OpenAI model access**

In [4]:
judge_invalid_api_key = Judge(
    openai="YOUR_API_KEY", anthropic=False, google=False, ollama=False
)

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: OpenAI!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               gemma4:e4b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


Providing an invalid API key (as here `"YOUR_API_KEY"`) will not throw an error upon creating a judge, but will obviously lead to failing requests.

**Please never commit your environments or API keys!**

### **Example: Setting up Ollama model access**

In [5]:
judge_gemma4 = Judge(openai=False, anthropic=False, google=False, ollama=True)

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: Ollama!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               gemma4:e4b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


You can setup the judge to use Ollama by providing `ollama=True` which will use the default model (currently `gemma4:e4b`) for rating translations.

In [6]:
judge_gemma4.config.ollama_model

'gemma4:e4b'

### **Example: Setting up a specific Ollama model**

In [7]:
judge_qwen = Judge(openai=False, anthropic=False, google=False, ollama="qwen3.8:27b")

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: Ollama!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               qwen3.8:27b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


You can request a specific Ollama model by providing the model identifier as the `ollama` argument.

### **Example: Using a custom configuration**

In [8]:
jc = JudgeConfig.model_validate_toml(toml_path="../config/judge_config.toml")
print(jc)

-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-6-astra
OpenAI Thinking Level:      medium
Anthropic Model:            claude-opus-5-5
Anthropic Thinking Level:   medium
Google Model:               gemini-3.8-flash
Google Thinking Level:      medium
Ollama Host:                http://localhost:11434
Ollama Model:               qwen3.8:27b
Ollama Keep Alive Duration: 5m
Maximum Output Tokens:      4096
Maximum Retries:            2
Retry Waiting Time:         10.0
Seeds:                      41467, 26, 77407, 1
-----------------------------------------------------


We can load a configuration from a config file using `JudgeConfig.model_validate_toml()`.

In [9]:
jc = JudgeConfig()
print(jc)

-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               gemma4:e4b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


Or just create the default configuration.

In [10]:
jc = JudgeConfig(ollama_model="mistral:7b")
print(jc)

-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               mistral:7b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


We can also manually setup the configuration, any unset configuration parameters will fallback to their defaults!

In [11]:
judge_with_config = Judge(
    openai=False, anthropic=False, google=False, ollama=True, config=jc
)

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: Ollama!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               mistral:7b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------


We can then create a `Judge` using our configuration. If no configuration is given (`config = None`, the default), then the default configuration will be used.

In [12]:
judge_with_config_and_ollama_model = Judge(
    openai=False, anthropic=False, google=False, ollama="qwen3.8:27b", config=jc
)

ERROR:llm_judge._judge:Parameter 'ollama' seems to be a model identifier but an Ollama model is already given in the configuration! Please only use one of the two options!


RuntimeError: Parameter 'ollama' seems to be a model identifier but an Ollama model is already given in the configuration! Please only use one of the two options!

When a configuration is explicitly passed, it is not allowed to also specify an Ollama model separately!

In [13]:
jr = judge_with_config.score(src=src, mt=mt, src_lang=src_lang, mt_lang=mt_lang)
jr.ollama.score  # ty: ignore[unresolved-attribute]

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:llm_judge._judge:Successfully got a valid response after retry 0 for one query.


0.9

We can then use the `Judge` for scoring translations...

In [14]:
judge_with_toml_config = Judge(
    openai=False,
    anthropic=False,
    google=False,
    ollama=True,
    config="../config/judge_config.toml",
)

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: Ollama!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-6-astra
OpenAI Thinking Level:      medium
Anthropic Model:            claude-opus-5-5
Anthropic Thinking Level:   medium
Google Model:               gemini-3.8-flash
Google Thinking Level:      medium
Ollama Host:                http://localhost:11434
Ollama Model:               qwen3.8:27b
Ollama Keep Alive Duration: 5m
Maximum Output Tokens:      4096
Maximum Retries:            2
Retry Waiting Time:         10.0
Seeds:                      41467, 26, 77407, 1
-----------------------------------------------------


We can also directly load a configuration from a TOML file with the `Judge`.

## **Scoring a Translation**

We can score a specific translation by using the `Judge.score()` function. We use the example data from above here.

In [15]:
jr = judge_qwen.score(src=src, mt=mt, src_lang=src_lang, mt_lang=mt_lang)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:llm_judge._judge:Successfully got a valid response after retry 0 for one query.


In [16]:
type(jr)

llm_judge._judge.JudgeResult

The `Judge.score()` function will return a `JudgeResult` which has four attributes (see also the docs on `JudgeResult` for details):
- `openai`
- `anthropic`
- `google`
- `ollama`

In [17]:
jr.openai is None

True

The `openai`, `anthropic` and `google` attributes will all be `None` because we did not setup the judge to use those models!

In [18]:
jr.ollama is None

False

The `ollama` attribute will not be `None` because we setup Ollama.

In [19]:
type(jr.ollama)

llm_judge._judge.JudgeModelResult

Any attribute of a `JudgeResult` will be a `JudgeModelResult` (if not `None`) - please also refer to the docs on `JudgeModelResult` for more details!

In [20]:
jr.ollama.model  # ty: ignore[unresolved-attribute]

'qwen3.8:27b'

Each `JudgeModelResult` contains the model.

In [21]:
jr.ollama.prompt  # ty: ignore[unresolved-attribute]

'You are an annotator for the quality of machine translation. Your task is to\n            identify errors and assess the quality of the translation.\n            Based on the source segment, human-generated reference translation, and machine\n            translation surrounded with triple backticks, identify error types in the\n            translation and classify them. The categories of errors are: accuracy\n            (addition, mistranslation, omission, untranslated text), fluency (character\n            encoding, grammar, inconsistency, punctuation, register, spelling), style\n            (awkward), terminology (inappropriate for context, inconsistent use),\n            non-translation, other, or no-error.\n            Each error is classified as one of three severities: critical, major, and minor.\n            Critical errors inhibit comprehension of the text. Major errors disrupt the\n            flow, but what the text is trying to say is still understandable. Minor errors\n  

Each `JudgeModelResult` contains the prompt that was used.

In [22]:
jr.ollama.status  # ty: ignore[unresolved-attribute]

'ok'

Each `JudgeModelResult` contains the status of the response which can be `"ok"` or `"error"`, please refer to the log to debug errors.

In [23]:
jr.ollama.parameters  # ty: ignore[unresolved-attribute]

{'num_predict': '2048', 'seed': '1337'}

Each `JudgeModelResult` contains the additional parameters that were used with the model.

In [24]:
jr.ollama.response  # ty: ignore[unresolved-attribute]

'model=\'qwen3.8:27b\' created_at=\'2026-09-27T18:54:33.1027934Z\' done=True done_reason=\'stop\' total_duration=75801302400 load_duration=7614530900 prompt_eval_count=394 prompt_eval_duration=1627347000 eval_count=1250 eval_duration=66554738000 message=Message(role=\'assistant\', content=\'{\\n  "source": "The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. I can see this feature as useful in an office setting with houseplants or if on vacation.",\\n  "machine_translation": "Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub.",\\n  "quality_estimation_value": 0.75,\\n  "errors": [\\n    {\\n      "error_type": "accuracy",\\n      "error_subtype": "mistranslation",\\n      "error_severity": "major",\\n      "error_index_start

Each `JudgeModelResult` with status `"ok"` contains the full model response.

In [25]:
type(jr.ollama.quality_estimation)  # ty: ignore[unresolved-attribute]

llm_judge._translation.QualityEstimation

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation`.

In [26]:
jr.ollama.quality_estimation_dict  # ty: ignore[unresolved-attribute]

{'source': 'The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. I can see this feature as useful in an office setting with houseplants or if on vacation.',
 'machine_translation': 'Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub.',
 'quality_estimation_value': 0.75,
 'errors': [{'error_type': 'accuracy',
   'error_subtype': 'mistranslation',
   'error_severity': 'major',
   'error_index_start': 1,
   'error_index_end': 3,
   'description': "The pronoun 'ihn' (him/it, masculine) refers to 'den Timer' in the German sentence, but in the source 'them' refers to 'the lights.' The source means the person prefers to turn the LIGHTS on and off manually, while the MT reads as if they prefer to turn the TIMER on and off themselves, 

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation` as python native dictionary.

In [27]:
jr.ollama.score  # ty: ignore[unresolved-attribute]

0.75

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation` score.

## **Scoring a Translation File**

We can also streamline the scoring of translations by submitting a `csv` file with the columns `src`, `mt`, `src_lang`, `mt_lang` and a judge to the utility function `annotate_csv()`.

In [28]:
pl_df, list_jr = annotate_csv("../data/test.csv", judge=judge_gemma4)

INFO:llm_judge._util:Reading file ../data/test.csv...
INFO:llm_judge._util:Successfully read file ../data/test.csv!
Annotating ../data/test.csv...:   0%|          | 0/1 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:llm_judge._judge:Successfully got a valid response after retry 0 for one query.
Annotating ../data/test.csv...: 100%|██████████| 1/1 [00:24<00:00, 24.04s/it]
INFO:llm_judge._util:Finished annotation of ../data/test.csv!


In [29]:
type(pl_df)

polars.dataframe.frame.DataFrame

In [30]:
pl_df

src,mt,src_lang,mt_lang,score_openai,score_anthropic,score_google,score_ollama_gemma4:e4b
str,str,str,str,f64,f64,f64,f64
"""The lights are dimmable, but I…","""Die Lichter sind dimmbar, aber…","""English""","""German""",NaN,NaN,NaN,0.98


This returns a polars DataFrame...

In [31]:
type(list_jr)

list

In [32]:
type(list_jr[0])

llm_judge._judge.JudgeResult

...and a list of `JudgeResult` containing the raw results.

## **Closing Connections and Clients**

It's good practice to close all the `Judge` instances once you are done via the `close()` method. This will close all LLM provider connections and clients.

In [33]:
judge_invalid_api_key.close()
judge_gemma4.close()
judge_qwen.close()
judge_with_config.close()
judge_with_toml_config.close()

INFO:llm_judge._judge:Successfully closed all connections and clients for this instance!
INFO:llm_judge._judge:Successfully closed all connections and clients for this instance!
INFO:llm_judge._judge:Successfully closed all connections and clients for this instance!
INFO:llm_judge._judge:Successfully closed all connections and clients for this instance!
INFO:llm_judge._judge:Successfully closed all connections and clients for this instance!


You can also use you a `Judge` instance with the `with` keywoard:

In [34]:
with Judge(
    openai=False, anthropic=False, google=False, ollama="mistral:7b"
) as automatically_closed_judge:
    pl_df, list_jr = annotate_csv("../data/test.csv", judge=automatically_closed_judge)

pl_df

INFO:llm_judge._judge:The following LLM-providers are enabled for this instance: Ollama!
INFO:llm_judge._judge:Loaded the following configuration:
-------------------- JudgeConfig --------------------
OpenAI Model:               gpt-5.4
OpenAI Thinking Level:      low
Anthropic Model:            claude-opus-5
Anthropic Thinking Level:   low
Google Model:               gemini-3.1-pro-preview
Google Thinking Level:      low
Ollama Host:                http://localhost:11434
Ollama Model:               mistral:7b
Ollama Keep Alive Duration: -1
Maximum Output Tokens:      2048
Maximum Retries:            5
Retry Waiting Time:         30.0
Seeds:                      1337, 10081995, 18041970, 1071966, 3082023, 24042025
-----------------------------------------------------
INFO:llm_judge._util:Reading file ../data/test.csv...
INFO:llm_judge._util:Successfully read file ../data/test.csv!
Annotating ../data/test.csv...:   0%|          | 0/1 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST http://

src,mt,src_lang,mt_lang,score_openai,score_anthropic,score_google,score_ollama_mistral:7b
str,str,str,str,f64,f64,f64,f64
"""The lights are dimmable, but I…","""Die Lichter sind dimmbar, aber…","""English""","""German""",NaN,NaN,NaN,0.9
